In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import time
import copy
import pickle

import numpy as np
import pandas as pd
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))


In [ ]:
def display_df(df):
    display(HTML(df.to_html()))
    return None

In [ ]:
def check_smiles_in(simles, smiles_lst):
    assert isinstance(simles, str)
    assert isinstance(smiles_lst, list)
    return simles.lower().strip() in [s.lower().strip() for s in smiles_lst]

# check in train

In [ ]:
df00 = pd.read_csv("data/process/pdY_ab42_train_test.csv")
df00["train_test"].value_counts()
SMILES_TRAIN = df00.loc[df00["train_test"] == "train", "smiles"].unique().tolist()
df00["in_train"] = df00["smiles"].apply(lambda s: check_smiles_in(s, SMILES_TRAIN)).astype(int)
df00.groupby(["train_test"])["in_train"].sum()

# test

In [ ]:
df00 = pd.read_csv("data/process/pdY_ab42_train_test.csv")
print("df00", df00.shape)
df00 = df00[df00["train_test"] == "test"]
print("df00", df00.shape)
display_df(df00.head())

cnames = ["smiles", "pred"]
df_rf = pd.read_csv("results/rf/train_test_pred.csv")
print("df_rf", df_rf.shape)
df_rf = df_rf.loc[df_rf["train_test"] == "test", cnames].rename(columns={"pred": "pred_rf"}).drop_duplicates(subset=["smiles"])
print("df_rf", df_rf.shape)
display_df(df_rf.head())



df_xb = pd.read_csv("results/xgb/train_test_pred.csv")
print("df_xb", df_xb.shape)
df_xb = df_xb.loc[df_xb["train_test"] == "test", cnames].rename(columns={"pred": "pred_xgb"}).drop_duplicates(subset=["smiles"])
print("df_xb", df_xb.shape)
display_df(df_xb.head())


df_gcov = pd.read_csv("results/graph/test.csv")
print("df_gcov", df_gcov.shape)
df_gcov = df_gcov[cnames].rename(columns={"pred": "pred_gconv"}).drop_duplicates(subset=["smiles"])
print("df_gcov", df_gcov.shape)
display_df(df_gcov.head())

df01 = (df00
        .merge(df_rf, how="inner", on="smiles")
        .merge(df_xb, how="inner", on="smiles")
        .merge(df_gcov, how="inner", on="smiles")
       )
df01["pred_ens"] = df01[["pred_rf", "pred_xgb", "pred_gconv"]].mean(axis=1)
print("df01", df01.shape)
display_df(df01.head())
df01.to_csv("results/ensemble/test.csv", index=False)

df01[["dG", "pred_rf", "pred_xgb", "pred_gconv", "pred_ens"]].corr()

In [ ]:
df01 = pd.read_csv("results/ensemble/test.csv")
print(df01.shape)

In [ ]:
df01.head()

In [ ]:
df01[["dG", "pred_rf", "pred_xgb", "pred_gconv", "pred_ens"]].corr()

## chembl33

In [ ]:
df00 = pd.read_csv("data/process/pdY_chembl33.csv")
print("df00", df00.shape)
display_df(df00.head())


cnames = ["smiles", "pred"]
df_rf = pd.read_csv("results/rf/chembl33_pred.csv")
print("df_rf", df_rf.shape)
df_rf = df_rf[cnames].rename(columns={"pred": "pred_rf"}).drop_duplicates(subset=["smiles"])
print("df_rf", df_rf.shape)
display_df(df_rf.head())


df_xb = pd.read_csv("results/xgb/chembl33_pred.csv")
print("df_xb", df_xb.shape)
df_xb = df_xb[cnames].rename(columns={"pred": "pred_xgb"}).drop_duplicates(subset=["smiles"])
print("df_xb", df_xb.shape)
display_df(df_xb.head())


df_gcov = pd.read_csv("results/graph/chembl33.csv")
print("df_gcov", df_gcov.shape)
df_gcov = df_gcov[cnames].rename(columns={"pred": "pred_gconv"}).drop_duplicates(subset=["smiles"])
print("df_gcov", df_gcov.shape)
display_df(df_gcov.head())

df01 = (df00
        .merge(df_rf, how="inner", on="smiles")
        .merge(df_xb, how="inner", on="smiles")
        .merge(df_gcov, how="inner", on="smiles")
       )
# # TODO: check fillna value
# df01["pred_gconv"] = df01["pred_gconv"].fillna(-5)

df01["pred_ens"] = df01[["pred_rf", "pred_xgb", "pred_gconv"]].mean(axis=1)
print("df01", df01.shape)
display_df(df01.head())


df01.to_csv("results/ensemble/chembl33_pred.csv", index=False)

df01[["pred_rf", "pred_xgb", "pred_gconv", "pred_ens"]].corr()

In [ ]:
df01.sort_values(by="pred_ens")

## chembl36excl33

In [ ]:
df00 = pd.read_csv("data/process/pdY_chembl36excl33.csv")
print("df00", df00.shape)
display_df(df00.head())


cnames = ["smiles", "pred"]
df_rf = pd.read_csv("results/rf/chembl36excl33_pred.csv")
print("df_rf", df_rf.shape)
df_rf = df_rf[cnames].rename(columns={"pred": "pred_rf"}).drop_duplicates(subset=["smiles"])
print("df_rf", df_rf.shape)
display_df(df_rf.head())


df_xb = pd.read_csv("results/xgb/chembl36excl33_pred.csv")
print("df_xb", df_xb.shape)
df_xb = df_xb[cnames].rename(columns={"pred": "pred_xgb"}).drop_duplicates(subset=["smiles"])
print("df_xb", df_xb.shape)
display_df(df_xb.head())


df_gcov = pd.read_csv("results/graph/chembl36excl33.csv")
print("df_gcov", df_gcov.shape)
df_gcov = df_gcov[cnames].rename(columns={"pred": "pred_gconv"}).drop_duplicates(subset=["smiles"])
print("df_gcov", df_gcov.shape)
display_df(df_gcov.head())

df01 = (df00
        .merge(df_rf, how="inner", on="smiles")
        .merge(df_xb, how="inner", on="smiles")
        .merge(df_gcov, how="inner", on="smiles")
       )
# # TODO: check fillna value
# df01["pred_gconv"] = df01["pred_gconv"].fillna(-5)

df01["pred_ens"] = df01[["pred_rf", "pred_xgb", "pred_gconv"]].mean(axis=1)
print("df01", df01.shape)
display_df(df01.head())


df01.to_csv("results/ensemble/chembl36excl33_pred.csv", index=False)

df01[["pred_rf", "pred_xgb", "pred_gconv", "pred_ens"]].corr()

## amylovis201

In [ ]:
df00 = pd.read_csv("data/process/pdY_amylovis201.csv")
print("df00", df00.shape)
display_df(df00.head())


cnames = ["smiles", "pred"]
df_rf = pd.read_csv("results/rf/amylovis201_pred.csv")
print("df_rf", df_rf.shape)
df_rf = df_rf[cnames].rename(columns={"pred": "pred_rf"}).drop_duplicates(subset=["smiles"])
print("df_rf", df_rf.shape)
display_df(df_rf.head())


df_xb = pd.read_csv("results/xgb/amylovis201_pred.csv")
print("df_xb", df_xb.shape)
df_xb = df_xb[cnames].rename(columns={"pred": "pred_xgb"}).drop_duplicates(subset=["smiles"])
print("df_xb", df_xb.shape)
display_df(df_xb.head())


df_gcov = pd.read_csv("results/graph/amylovis201.csv")
print("df_gcov", df_gcov.shape)
df_gcov = df_gcov[cnames].rename(columns={"pred": "pred_gconv"}).drop_duplicates(subset=["smiles"])
print("df_gcov", df_gcov.shape)
display_df(df_gcov.head())

df01 = (df00
        .merge(df_rf, how="inner", on="smiles")
        .merge(df_xb, how="inner", on="smiles")
        .merge(df_gcov, how="inner", on="smiles")
       )
# # TODO: check fillna value
# df01["pred_gconv"] = df01["pred_gconv"].fillna(-5)

df01["pred_ens"] = df01[["pred_rf", "pred_xgb", "pred_gconv"]].mean(axis=1)
print("df01", df01.shape)
display_df(df01.head())


df01.to_csv("results/ensemble/amylovis201_pred.csv", index=False)

df01[["pred_rf", "pred_xgb", "pred_gconv", "pred_ens"]].corr()

## ref01

In [ ]:
df00 = pd.read_csv("data/process/pdY_ref01.csv")
print("df00", df00.shape)
display_df(df00.head())


cnames = ["smiles", "pred"]
df_rf = pd.read_csv("results/rf/ref01_pred.csv")
print("df_rf", df_rf.shape)
df_rf = df_rf[cnames].rename(columns={"pred": "pred_rf"}).drop_duplicates(subset=["smiles"])
print("df_rf", df_rf.shape)
display_df(df_rf.head())


df_xb = pd.read_csv("results/xgb/ref01_pred.csv")
print("df_xb", df_xb.shape)
df_xb = df_xb[cnames].rename(columns={"pred": "pred_xgb"}).drop_duplicates(subset=["smiles"])
print("df_xb", df_xb.shape)
display_df(df_xb.head())


df_gcov = pd.read_csv("results/graph/ref01.csv")
print("df_gcov", df_gcov.shape)
df_gcov = df_gcov[cnames].rename(columns={"pred": "pred_gconv"}).drop_duplicates(subset=["smiles"])
print("df_gcov", df_gcov.shape)
display_df(df_gcov.head())

df01 = (df00
        .merge(df_rf, how="inner", on="smiles")
        .merge(df_xb, how="inner", on="smiles")
        .merge(df_gcov, how="inner", on="smiles")
       )
# # TODO: check fillna value
# df01["pred_gconv"] = df01["pred_gconv"].fillna(-5)

df01["pred_ens"] = df01[["pred_rf", "pred_xgb", "pred_gconv"]].mean(axis=1)
print("df01", df01.shape)
display_df(df01.head())


df01.to_csv("results/ensemble/ref01_pred.csv", index=False)

df01[["pred_rf", "pred_xgb", "pred_gconv", "pred_ens"]].corr()